# **Day 39: Stacking and Blending**
*Module 6 - Ensemble Learning*

Voting averages models with fixed weights. **Stacking** (stacked generalisation, Wolpert 1992) **learns** how to combine them: a **meta-model** is trained on the base models' predictions. It is the standard final step of winning competition solutions and is increasingly used in remote sensing for combining RF, SVM, boosting and neural networks.

> Stacking trains a "meta-model" that learns how best to combine the predictions of several different base models. Blending is a simpler version that uses a hold-out set.
>
> *Example:* An RF, an SVM and a neural net each predict crop type, and a logistic regression learns that the SVM is most reliable for wheat.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression, RidgeCV, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, r2_score
rng = np.random.default_rng(39)

## **1. The Leakage Trap and Out-of-Fold Predictions**
If base models predict their **own training data**, their predictions are over-confident (a deep Random Forest predicts training labels almost perfectly). A meta-model trained on these predictions learns to **trust the most overfitted model**. Solution: train the meta-model on **out-of-fold** predictions: each training sample is predicted by a base model that did **not** see it.

In [2]:
from sklearn.datasets import make_friedman1
X, y = make_friedman1(3000, noise=1.5, random_state=0)
X = np.c_[X, rng.normal(size=(3000, 10))]                         # add noise features
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
base = {"ridge": RidgeCV(), "knn": make_pipeline(StandardScaler(), KNeighborsRegressor(10)),
        "rf": RandomForestRegressor(300, n_jobs=-1, random_state=0), "hgb": HistGradientBoostingRegressor(random_state=0),
        "svr": make_pipeline(StandardScaler(), SVR(C=10))}

# WRONG: in-sample predictions as meta-features
in_sample = np.column_stack([m.fit(X_tr, y_tr).predict(X_tr) for m in base.values()])
test_preds = np.column_stack([m.predict(X_te) for m in base.values()])
meta_wrong = LinearRegression().fit(in_sample, y_tr)
# RIGHT: out-of-fold predictions
kf = KFold(5, shuffle=True, random_state=0)
oof = np.column_stack([cross_val_predict(m, X_tr, y_tr, cv=kf) for m in base.values()])
meta_right = LinearRegression().fit(oof, y_tr)
print("meta weights (in-sample, leaky):", dict(zip(base, meta_wrong.coef_.round(2).tolist())))
print("meta weights (out-of-fold)     :", dict(zip(base, meta_right.coef_.round(2).tolist())))
print(f"test R2: leaky stack {r2_score(y_te, meta_wrong.predict(test_preds)):.4f} | OOF stack {r2_score(y_te, meta_right.predict(test_preds)):.4f}")
for name, col in zip(base, test_preds.T):
    print(f"   base {name:<5} test R2 {r2_score(y_te, col):.4f}")

meta weights (in-sample, leaky): {'ridge': -0.14, 'knn': -0.04, 'rf': 0.53, 'hgb': 0.37, 'svr': 0.27}
meta weights (out-of-fold)     : {'ridge': 0.03, 'knn': -0.03, 'rf': -0.21, 'hgb': 1.08, 'svr': 0.15}
test R2: leaky stack 0.8681 | OOF stack 0.8849
   base ridge test R2 0.7039
   base knn   test R2 0.5474
   base rf    test R2 0.8193
   base hgb   test R2 0.8818
   base svr   test R2 0.8158


The leaky meta-model gives most weight to the Random Forest (which memorises training data). The OOF meta-model weights models by their **real** generalisation.

## **2. Stacking From Scratch (Classification)**

In [3]:
class StackingScratch:
    def __init__(self, base_models, meta_model, n_splits=5, seed=0):
        self.base_models, self.meta_model, self.n_splits, self.seed = base_models, meta_model, n_splits, seed
    def fit(self, X, y):
        from sklearn.base import clone
        cv = StratifiedKFold(self.n_splits, shuffle=True, random_state=self.seed)
        oof = np.zeros((len(X), len(self.base_models)))
        for j, m in enumerate(self.base_models):
            for tr, va in cv.split(X, y):
                oof[va, j] = clone(m).fit(X[tr], y[tr]).predict_proba(X[va])[:, 1]
        self.fitted_ = [clone(m).fit(X, y) for m in self.base_models]       # refit on all data for test time
        self.meta_ = clone(self.meta_model).fit(oof, y)
        return self
    def predict_proba(self, X):
        Z = np.column_stack([m.predict_proba(X)[:, 1] for m in self.fitted_])
        return self.meta_.predict_proba(Z)

from sklearn.datasets import make_classification
Xc, yc = make_classification(4000, 30, n_informative=10, n_redundant=5, flip_y=0.05, class_sep=0.7, random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, stratify=yc, random_state=0)
bases_c = [make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), RandomForestClassifier(300, n_jobs=-1, random_state=0),
           HistGradientBoostingClassifier(random_state=0), make_pipeline(StandardScaler(), KNeighborsClassifier(25))]
names_c = ["logreg", "rf", "hgb", "knn"]
for n_, m in zip(names_c, bases_c):
    print(f"{n_:<7} test AUC {roc_auc_score(yc_te, m.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1]):.4f}")
st = StackingScratch(bases_c, LogisticRegression()).fit(Xc_tr, yc_tr)
print(f"stack   test AUC {roc_auc_score(yc_te, st.predict_proba(Xc_te)[:, 1]):.4f} | meta weights {dict(zip(names_c, st.meta_.coef_[0].round(2).tolist()))}")
avg = np.mean([m.predict_proba(Xc_te)[:, 1] for m in bases_c], axis=0)
print(f"simple average  AUC {roc_auc_score(yc_te, avg):.4f}")

logreg  test AUC 0.8600


rf      test AUC 0.9526


hgb     test AUC 0.9622
knn     test AUC 0.9401


stack   test AUC 0.9627 | meta weights {'logreg': -1.67, 'rf': 2.38, 'hgb': 4.5, 'knn': 2.52}
simple average  AUC 0.9531


## **3. scikit-learn's StackingClassifier**
Handles the OOF logic internally (`cv=5`). `passthrough=True` also gives the original features to the meta-model.

In [4]:
from sklearn.ensemble import StackingClassifier, StackingRegressor
stack_sk = StackingClassifier(estimators=list(zip(names_c, bases_c)), final_estimator=LogisticRegression(), cv=5,
                              stack_method="predict_proba", n_jobs=-1)
stack_sk.fit(Xc_tr, yc_tr)
print(f"StackingClassifier test AUC {roc_auc_score(yc_te, stack_sk.predict_proba(Xc_te)[:, 1]):.4f}")

stack_reg = StackingRegressor(list(base.items()), final_estimator=RidgeCV(), cv=5, n_jobs=-1).fit(X_tr, y_tr)
print(f"StackingRegressor  test R2  {stack_reg.score(X_te, y_te):.4f}")

StackingClassifier test AUC 0.9623


StackingRegressor  test R2  0.8849


## **4. Blending: The Hold-Out Variant**
Split training data into A and B. Train base models on A, predict B; train the meta-model on B's predictions. Simpler and faster, but each stage uses less data.

In [5]:
XA, XB, yA, yB = train_test_split(Xc_tr, yc_tr, test_size=0.3, stratify=yc_tr, random_state=0)
from sklearn.base import clone
fitted = [clone(m).fit(XA, yA) for m in bases_c]
meta_b = LogisticRegression().fit(np.column_stack([m.predict_proba(XB)[:, 1] for m in fitted]), yB)
p_blend = meta_b.predict_proba(np.column_stack([m.predict_proba(Xc_te)[:, 1] for m in fitted]))[:, 1]
print(f"blending test AUC {roc_auc_score(yc_te, p_blend):.4f}")

blending test AUC 0.9587


## **5. Design Rules**
1. **Diverse base models** (linear, kernel, trees, neighbours, neural nets) beat many similar ones.
2. **Simple, regularised meta-model**: logistic/ridge regression, often with non-negative weights. A complex meta-model overfits the OOF predictions.
3. Use **probabilities** (or log-odds), not hard labels, as meta-features.
4. Evaluate the whole stack with an **outer** CV or a test set never used in stacking.
5. Report the gain over the best single model; stacking often adds only 0.5-2%.

# **Part 2: Going Deeper - The Super Learner**

The **Super Learner** (van der Laan et al., 2007) is stacking with a **convex combination** meta-model: non-negative weights summing to 1, fitted to OOF predictions. It has an oracle property: asymptotically it performs as well as the best possible weighted combination of the base models.

In [6]:
from scipy.optimize import nnls
def super_learner_weights(oof_preds, y):
    w, _ = nnls(oof_preds, y)                  # non-negative least squares
    return w / w.sum()

w_sl = super_learner_weights(oof, y_tr)
print("Super Learner weights:", dict(zip(base, w_sl.round(3).tolist())))
print(f"Super Learner test R2: {r2_score(y_te, test_preds @ w_sl):.4f}")

Super Learner weights: {'ridge': 0.0, 'knn': 0.0, 'rf': 0.0, 'hgb': 0.912, 'svr': 0.088}
Super Learner test R2: 0.8834


### Is the gain real? Outer cross-validation of the whole procedure

In [7]:
from sklearn.model_selection import cross_validate
outer = KFold(5, shuffle=True, random_state=42)
small = {k: base[k] for k in ["ridge", "rf", "hgb", "svr"]}
best_single = cross_val_score(HistGradientBoostingRegressor(random_state=0), X, y, cv=outer).mean()
stacked = cross_val_score(StackingRegressor(list(small.items()), final_estimator=RidgeCV(), cv=5, n_jobs=-1), X, y, cv=outer)
print(f"best single (HGB) outer-CV R2 {best_single:.4f} | stack {stacked.mean():.4f} +/- {stacked.std():.4f}")

best single (HGB) outer-CV R2 0.8835 | stack 0.8865 +/- 0.0109


## **Practice Problems**
1. Add `passthrough=True` to `StackingClassifier`. Does the meta-model benefit from the raw features?
2. Replace the meta-model by a `RandomForestClassifier`. What happens to test AUC (overfitting of the meta-level)?
3. Remove the best base model from the stack. How much AUC is lost? What does that say about redundancy?
4. *(Advanced)* Build a 2-level stack: level 1 = 4 base models; level 2 = two different meta-models (logistic, HGB); level 3 = average of level-2 outputs. Evaluate on the test set.

In [8]:
# Solution 1
s_pt = StackingClassifier(list(zip(names_c, bases_c)), LogisticRegression(max_iter=2000), cv=5, passthrough=True, n_jobs=-1).fit(Xc_tr, yc_tr)
print("passthrough AUC:", round(roc_auc_score(yc_te, s_pt.predict_proba(Xc_te)[:, 1]), 4))
# Solution 2
s_rf = StackingClassifier(list(zip(names_c, bases_c)), RandomForestClassifier(300, random_state=0), cv=5, n_jobs=-1).fit(Xc_tr, yc_tr)
print("RF meta-model AUC:", round(roc_auc_score(yc_te, s_rf.predict_proba(Xc_te)[:, 1]), 4))
# Solution 3
best_idx = int(np.argmax([roc_auc_score(yc_te, m.predict_proba(Xc_te)[:, 1]) for m in bases_c]))
rest = [(n_, m) for i, (n_, m) in enumerate(zip(names_c, bases_c)) if i != best_idx]
s_rest = StackingClassifier(rest, LogisticRegression(), cv=5, n_jobs=-1).fit(Xc_tr, yc_tr)
print(f"without {names_c[best_idx]}: AUC {roc_auc_score(yc_te, s_rest.predict_proba(Xc_te)[:, 1]):.4f}")
# Solution 4
lvl2 = [StackingClassifier(list(zip(names_c, bases_c)), fe, cv=5, n_jobs=-1).fit(Xc_tr, yc_tr)
        for fe in [LogisticRegression(), HistGradientBoostingClassifier(max_iter=100, random_state=0)]]
p3 = np.mean([m.predict_proba(Xc_te)[:, 1] for m in lvl2], axis=0)
print("3-level stack AUC:", round(roc_auc_score(yc_te, p3), 4))

passthrough AUC: 0.9612


RF meta-model AUC: 0.9533


without hgb: AUC 0.9555


3-level stack AUC: 0.9585


## **Key References**
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5(2), 241-259.
- Breiman, L. (1996). Stacked regressions. *Machine Learning*, 24(1), 49-64.
- van der Laan, M. J., Polley, E. C., & Hubbard, A. E. (2007). Super learner. *Statistical Applications in Genetics and Molecular Biology*, 6(1), Article 25.
- Ting, K. M., & Witten, I. H. (1999). Issues in stacked generalization. *Journal of Artificial Intelligence Research*, 10, 271-289.